# Lag Contribution Per Touch

Rank UFA players by how much value they set up for the next thrower.

## tl;dr

Shown Space's official `LC` is not included as a raw column in the cached throw data. This notebook builds a transparent proxy: each thrower gets credit for the `aec` of the next throw in the same possession. That makes the metric a "did this touch set up the next valuable action?" leaderboard.

Use `MIN_TOUCHES` to control sample size. The default is 100 thrower touches.

## Context & Method

`lag_contribution` is credited to the current thrower as the next throw's `aec` within the same possession. The final throw of a possession receives zero lag contribution because it does not set up another throw.

In this notebook, a `touch` means a throw attempt by the player. That matches the row grain of the Shown Space throw data and keeps the denominator clear.

## Setup

In [1]:
import importlib
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd
import plotly.express as px

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import ufa_aec_possessions
import ufa_aec_possessions.fetching
import ufa_aec_possessions.players

importlib.reload(ufa_aec_possessions.fetching)
importlib.reload(ufa_aec_possessions.players)
importlib.reload(ufa_aec_possessions)

from ufa_aec_possessions import (
    fetch_shownspace_season_throws_cached,
    summarize_lag_contribution_per_touch,
)

In [2]:
SEASON = 2026
MAX_GAMES = None
FORCE_REFRESH = False
CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'shownspace'

MIN_TOUCHES = 100
TOP_N = 25

## Data

In [3]:
league_games, league_throws = fetch_shownspace_season_throws_cached(
    season=SEASON,
    max_games=MAX_GAMES,
    cache_dir=CACHE_DIR,
    force_refresh=FORCE_REFRESH,
)

print(f'League games loaded: {len(league_games):,}')
print(f'League throws loaded: {len(league_throws):,}')

League games loaded: 118
League throws loaded: 64,964


## Leaderboard

In [4]:
lag_leaderboard = summarize_lag_contribution_per_touch(
    league_throws,
    min_touches=MIN_TOUCHES,
)

display_columns = [
    'rank', 'team_id', 'thrower', 'lag_contribution_per_touch',
    'lag_contribution_per_100_touches', 'total_lag_contribution', 'touches',
    'goal_setups', 'goal_setup_rate', 'thrower_aec_per_touch',
    'completion_rate', 'avg_cp', 'avg_cpoe', 'huck_rate', 'avg_throw_distance',
]
lag_table = lag_leaderboard.reindex(columns=display_columns).copy()
for column in [
    'lag_contribution_per_touch', 'lag_contribution_per_100_touches',
    'total_lag_contribution', 'goal_setup_rate', 'thrower_aec_per_touch',
    'completion_rate', 'avg_cp', 'avg_cpoe', 'huck_rate', 'avg_throw_distance',
]:
    lag_table[column] = pd.to_numeric(lag_table[column], errors='coerce').round(3)
lag_table['team_id'] = lag_table['team_id'].astype(str).str.title()
lag_table.head(TOP_N)

,rank,team_id,thrower,lag_contribution_per_touch,lag_contribution_per_100_touches,total_lag_contribution,touches,goal_setups,goal_setup_rate,thrower_aec_per_touch,completion_rate,avg_cp,avg_cpoe,huck_rate,avg_throw_distance
0,1,Empire,bjagt,0.098,9.808,12.555,128,20,0.156,0.102,1.0,0.943,0.041,0.062,17.186
1,2,Flyers,zthoreson,0.086,8.645,12.968,150,17,0.113,0.120,1.0,0.935,0.045,0.087,18.023
2,3,Alleycats,sgudeman,0.086,8.640,10.109,117,14,0.120,0.021,1.0,0.945,0.020,0.043,14.616
3,4,Breeze,jwodatch1,0.086,8.639,10.194,118,15,0.127,0.044,1.0,0.933,0.024,0.051,18.081
4,5,Empire,bsimmons,0.084,8.427,9.185,109,13,0.119,0.037,1.0,0.969,0.003,0.000,14.156
5,6,Cascades,zraunig,0.082,8.248,9.073,110,15,0.136,0.085,1.0,0.931,0.060,0.009,13.766
6,7,Hustle,bhulsmeye,0.080,8.046,16.011,199,15,0.075,0.027,1.0,0.955,0.002,0.010,15.410
7,8,Shred,jduennebe,0.078,7.834,9.715,124,13,0.105,0.055,1.0,0.915,0.032,0.121,19.851
8,9,Empire,ochartock,0.078,7.762,8.849,114,9,0.079,0.061,1.0,0.955,0.010,0.035,16.017
9,10,Spiders,dwhited,0.078,7.762,7.995,103,12,0.117,0.093,1.0,0.950,0.030,0.019,13.894


In [5]:
plot_data = lag_leaderboard.head(TOP_N).sort_values('lag_contribution_per_touch', ascending=True).copy()
plot_data['player_label'] = (
    plot_data['rank'].astype(int).astype(str)
    + '. '
    + plot_data['thrower'].astype(str)
    + ' ('
    + plot_data['team_id'].astype(str).str.title()
    + ')'
)

fig = px.bar(
    plot_data,
    x='lag_contribution_per_touch',
    y='player_label',
    orientation='h',
    text='lag_contribution_per_touch',
    color='lag_contribution_per_touch',
    color_continuous_scale=[(0, '#eaf4e8'), (0.5, '#6fbe73'), (1, '#09552f')],
    hover_data={
        'player_label': False,
        'lag_contribution_per_touch': ':.3f',
        'lag_contribution_per_100_touches': ':.2f',
        'total_lag_contribution': ':.2f',
        'touches': ':,',
        'goal_setups': ':,',
        'goal_setup_rate': ':.1%',
        'thrower_aec_per_touch': ':.3f',
        'avg_cp': ':.1%',
        'avg_cpoe': ':.1%',
        'huck_rate': ':.1%',
    },
    labels={
        'lag_contribution_per_touch': 'lag contribution per touch',
        'player_label': '',
    },
    title=f'{SEASON} UFA lag contribution per touch leaders, min {MIN_TOUCHES} touches',
)
fig.update_traces(texttemplate='%{x:.3f}', textposition='outside', cliponaxis=False)
fig.update_layout(
    title={'x': 0.5, 'xanchor': 'center'},
    width=1040,
    height=max(560, 28 * len(plot_data) + 160),
    margin={'l': 190, 'r': 90, 't': 76, 'b': 64},
    coloraxis_showscale=False,
    plot_bgcolor='#f7fbf5',
    paper_bgcolor='#ffffff',
    font={'family': 'Segoe UI, Arial, sans-serif', 'color': '#20385f'},
    hoverlabel={'bgcolor': '#ffffff', 'font': {'color': '#14213d'}},
)
fig.update_xaxes(showgrid=True, gridcolor='#e5eee2', zeroline=False)
fig.update_yaxes(ticks='')
fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'cliponaxis': False,
              'customdata': {'bdata': ('Dz2BXUYFGkBumb9RgXkyQAAAAAAAwH' ... '6tRRwv7j/mEiWlOw6lPwAAAAAAALA/'),
                             'dtype': 'f8',
                             'shape': '25, 9'},
              'hovertemplate': ('lag contribution per touch=%{m' ... 'tomdata[8]:.1%}<extra></extra>'),
              'legendgroup': '',
              'marker': {'color': {'bdata': ('zEUVRjensD8kMSTNsL+wPyDExvkfAr' ... 'x3oR62P3LlHDeZIbY/42l3kg0cuT8='),
                                   'dtype': 'f8'},
                         'coloraxis': 'coloraxis',
                         'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'h',
              'showlegend': False,
              'text': {'bdata': ('zEUVRjensD8kMSTNsL+wPyDExvkfAr' ... 'x3oR62P3LlHDeZIbY/42l3kg0cuT8='),
                       'dtype': 'f8'},
              'textposition': 'outside',
              'texttemplate': '%{x:.3f}',
              'type': 'bar',
              'x': {'bdata': ('zEUVRjensD8kMSTNsL+wPyDExvkfAr' ... 'x3oR62P3LlHDeZIbY/42l3kg0cuT8='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['25. mmiller1 (Growlers)', '24. marmstron (Sol)',
                          '23. amerriman (Breeze)', '22. srueschem (Empire)',
                          '21. nhanson (Windchill)', '20. arousseau (Royal)',
                          '19. wstpierre (Royal)', '18. ichang (Spiders)',
                          '17. jfairfax (Flyers)', '16. mbrownlee (Empire)',
                          '15. mlabar (Empire)', '14. rpieran (Radicals)',
                          '13. jwilliams (Empire)', '12. lmcclamro (Hustle)',
                          '11. gsanner (Flyers)', '10. dwhited (Spiders)',
                          '9. ochartock (Empire)', '8. jduennebe (Shred)',
                          '7. bhulsmeye (Hustle)', '6. zraunig (Cascades)',
                          '5. bsimmons (Empire)', '4. jwodatch1 (Breeze)',
                          '3. sgudeman (Alleycats)', '2. zthoreson (Flyers)', '1. bjagt (Empire)'],
                         dtype=object),
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'coloraxis': {'autocolorscale': False,
                             'colorbar': {'title': {'text': 'lag contribution per touch'}},
                             'colorscale': [[0, '#eaf4e8'], [0.5, '#6fbe73'], [1,
                                            '#09552f']],
                             'showscale': False},
               'font': {'color': '#20385f', 'family': 'Segoe UI, Arial, sans-serif'},
               'height': 860,
               'hoverlabel': {'bgcolor': '#ffffff', 'font': {'color': '#14213d'}},
               'legend': {'tracegroupgap': 0},
               'margin': {'b': 64, 'l': 190, 'r': 90, 't': 76},
               'paper_bgcolor': '#ffffff',
               'plot_bgcolor': '#f7fbf5',
               'template': '...',
               'title': {'text': '2026 UFA lag contribution per touch leaders, min 100 touches',
                         'x': 0.5,
                         'xanchor': 'center'},
               'width': 1040,
               'xaxis': {'anchor': 'y',
                         'domain': [0.0, 1.0],
                         'gridcolor': '#e5eee2',
                         'showgrid': True,
                         'title': {'text': 'lag contribution per touch'},
                         'zeroline': False},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'ticks': '', 'title': {'text': ''}}}
})

## Volume Check

Per-touch leaderboards can favor smaller samples. This table shows the total lag contribution leaders so we can separate efficiency from total workload.

In [6]:
volume_table = lag_leaderboard.sort_values('total_lag_contribution', ascending=False).head(TOP_N).copy()
volume_table = volume_table.reindex(columns=display_columns)
for column in [
    'lag_contribution_per_touch', 'lag_contribution_per_100_touches',
    'total_lag_contribution', 'goal_setup_rate', 'thrower_aec_per_touch',
    'completion_rate', 'avg_cp', 'avg_cpoe', 'huck_rate', 'avg_throw_distance',
]:
    volume_table[column] = pd.to_numeric(volume_table[column], errors='coerce').round(3)
volume_table['team_id'] = volume_table['team_id'].astype(str).str.title()
volume_table

,rank,team_id,thrower,lag_contribution_per_touch,lag_contribution_per_100_touches,total_lag_contribution,touches,goal_setups,goal_setup_rate,thrower_aec_per_touch,completion_rate,avg_cp,avg_cpoe,huck_rate,avg_throw_distance
61,62,Shred,myorgason,0.050,4.977,23.145,465,26,0.056,0.035,1.0,0.962,0.010,0.017,14.619
36,37,Flyers,dhawkins,0.058,5.809,22.306,384,24,0.062,0.035,1.0,0.957,0.017,0.023,15.706
40,41,Radicals,gvordick,0.057,5.679,21.921,386,24,0.062,0.036,1.0,0.957,0.014,0.013,15.694
56,57,Flyers,jlouie,0.051,5.123,21.823,426,29,0.068,0.054,1.0,0.968,0.018,0.009,14.146
53,54,Breeze,jnissen,0.052,5.165,21.642,419,29,0.069,0.077,1.0,0.943,0.036,0.050,18.987
68,69,Hustle,amiller1,0.048,4.841,20.720,428,37,0.086,0.076,1.0,0.960,0.015,0.028,15.528
87,88,Apex,telliott,0.042,4.239,20.345,480,25,0.052,0.022,1.0,0.962,-0.010,0.021,16.045
85,86,Windchill,glarson,0.042,4.249,19.037,448,28,0.062,0.084,1.0,0.948,0.019,0.060,18.106
24,25,Growlers,mmiller1,0.065,6.505,18.475,284,17,0.060,0.054,1.0,0.930,0.042,0.095,19.132
14,15,Empire,mlabar,0.073,7.277,18.048,248,22,0.089,0.065,1.0,0.941,0.002,0.052,17.372


## Takeaways

After running the notebook, compare the rate and volume tables:

- The rate leaderboard answers: who sets up the most next-throw value per touch?
- The volume table answers: who creates the most total next-throw value over the season?
- Players high in both views are the strongest candidates for a Shown Space-style "facilitator" story.